# Aula 3 — Transfer Learning com PyTorch

Notebook de acompanhamento da Aula 3 — **Tópicos Avançados em IA · UFABC**  
Adaptado de MIT 15.773 (Farias & Ramakrishnan, 2024)

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ufabc-bcc/deep-learning-course/blob/main/notebooks/lec03-torch.ipynb)

---

## Índice

1. [Introdução](#1-introdução)
2. [Configuração](#2-configuração)
3. [CIFAR-10](#3-cifar-10)
4. [Fase 1 — Feature Extraction](#4-fase-1--feature-extraction)
5. [Fase 2 — Fine-tuning](#5-fase-2--fine-tuning)
6. [Resultados](#6-resultados)
7. [Predições](#7-predições)

## 1. Introdução

### O que é Transfer Learning?

Treinar uma rede do zero exige **milhões de imagens** e dias de computação.  
**Transfer Learning** reutiliza uma rede já treinada em uma tarefa grande (e.g. ImageNet)  
e a adapta para uma tarefa menor com muito menos dados e tempo.

A ideia central:

> As primeiras camadas de uma CNN aprendem **características universais**  
> (bordas, texturas, formas) que são úteis em qualquer domínio visual.

---

### Duas estratégias principais

| Estratégia | O que faz | Quando usar |
|---|---|---|
| **Feature Extraction** | Congela todo o backbone; treina só o classificador final | Dataset pequeno |
| **Fine-tuning** | Descongela parte ou todo o backbone; treina com lr baixo | Dataset médio/grande |

Neste notebook usaremos as duas em sequência:

$$
\underbrace{\mathscr{F}_{\text{ResNet18}}(x)}_\text{backbone congelado}
\;\longrightarrow\;
\underbrace{W \cdot h + b}_\text{nova cabeça}
\;\longrightarrow\;
\hat{y} \in \mathbb{R}^{10}
$$

**Dataset**: CIFAR-10 — 60 000 imagens 32×32, 10 classes.

## 2. Configuração

In [ ]:
# @title Instalar dependências (necessário no Colab)
# Descomente se estiver no Colab
# !pip install -q torch torchvision

In [ ]:
# @title Imports
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Usando device: {device}')

## 3. CIFAR-10

Fazemos `Resize(96)` para que a entrada seja maior que 32×32 (ResNet foi treinada com 224×224);  
usamos as **estatísticas do ImageNet** para normalização — assim os pesos pré-treinados ficam
"em casa" desde o primeiro forward pass.

In [ ]:
# @title Transformações e DataLoaders
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

transform_train = transforms.Compose([
    transforms.Resize(96),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

transform_test = transforms.Compose([
    transforms.Resize(96),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_ds = torchvision.datasets.CIFAR10(root='./data', train=True,
                                         download=True, transform=transform_train)
test_ds  = torchvision.datasets.CIFAR10(root='./data', train=False,
                                         download=True, transform=transform_test)

# Separa 10% do treino como validação
val_size   = int(0.1 * len(train_ds))
train_size = len(train_ds) - val_size
train_ds, val_ds = torch.utils.data.random_split(
    train_ds, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

CLASS_NAMES = ['avião','automóvel','pássaro','gato','cervo',
               'cachorro','sapo','cavalo','navio','caminhão']

print(f'Treino: {len(train_ds):,}  | Val: {len(val_ds):,}  | Teste: {len(test_ds):,}')

In [ ]:
# @title Visualizar amostras (desnormalizadas)
def denormalize(tensor, mean=MEAN, std=STD):
    """Reverte a normalização ImageNet para exibição."""
    t = tensor.clone().cpu()
    for c, m, s in zip(range(3), mean, std):
        t[c] = t[c] * s + m
    return t.permute(1, 2, 0).clamp(0, 1).numpy()

imgs, lbls = next(iter(train_loader))

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(denormalize(imgs[i]))
    ax.set_title(CLASS_NAMES[lbls[i].item()])
    ax.axis('off')
plt.suptitle('Amostras do CIFAR-10 (desnormalizadas)')
plt.tight_layout()
plt.show()

## 4. Fase 1 — Feature Extraction

Carregamos a **ResNet-18** pré-treinada no ImageNet e **congelamos todo o backbone**.  
Substituímos apenas a última camada fully-connected (`fc`) por uma linear com 10 saídas.  
Assim apenas ~5 120 parâmetros são treinados — o backbone vira um extrator de features fixo.

```
backbone (congelado)  →  pool  →  fc (512 → 10, treinável)
```

In [ ]:
# @title Carregar ResNet-18 pré-treinada e congelar backbone
base = models.resnet18(weights='IMAGENET1K_V1')

# Congela todos os parâmetros
for p in base.parameters():
    p.requires_grad = False

# Substitui a cabeça de classificação
base.fc = nn.Linear(base.fc.in_features, 10)

base = base.to(device)

trainable = sum(p.numel() for p in base.parameters() if p.requires_grad)
total      = sum(p.numel() for p in base.parameters())
print(f'Parâmetros treináveis: {trainable:,} / {total:,}  ({100*trainable/total:.2f}%)')

In [ ]:
# @title Função de treino reutilizável
def train_one_epoch(model, loader, criterion, optimizer):
    """Treina por uma época; retorna (loss_médio, acurácia)."""
    model.train()
    total_loss = correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out  = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct    += (out.argmax(1) == y).sum().item()
        total      += x.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    """Avalia sem gradiente; retorna (loss_médio, acurácia)."""
    model.eval()
    total_loss = correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out  = model(x)
        loss = criterion(out, y)
        total_loss += loss.item() * x.size(0)
        correct    += (out.argmax(1) == y).sum().item()
        total      += x.size(0)
    return total_loss / total, correct / total


def run_training(model, train_loader, val_loader, optimizer, criterion, n_epochs, tag=''):
    """Loop de treino completo; retorna histórico de acurácias."""
    train_accs, val_accs = [], []
    for epoch in range(1, n_epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        vl_loss, vl_acc = evaluate(model, val_loader, criterion)
        train_accs.append(tr_acc)
        val_accs.append(vl_acc)
        print(f"{tag} Época {epoch:2d}/{n_epochs} "
              f"— loss: {tr_loss:.4f}  acc treino: {tr_acc:.1%}  acc val: {vl_acc:.1%}")
    return train_accs, val_accs

In [ ]:
# @title Treinar Fase 1 (5 épocas)
criterion  = nn.CrossEntropyLoss()
optimizer1 = optim.Adam(base.fc.parameters(), lr=1e-3)

N_EPOCHS_PHASE1 = 5
tr_acc_p1, vl_acc_p1 = run_training(
    base, train_loader, val_loader,
    optimizer1, criterion,
    N_EPOCHS_PHASE1, tag='[Fase 1]'
)
print(f'\nAcurácia de validação ao fim da Fase 1: {vl_acc_p1[-1]:.1%}')

## 5. Fase 2 — Fine-tuning

Descongelamos **todos** os parâmetros e retreinamos com uma **taxa de aprendizado muito menor**  
(`1e-4`).  Isso evita que os pesos do ImageNet sejam destruídos por atualizações agressivas.

> **Por que lr menor?**  Os pesos do backbone já são bons pontos de partida.  
> Uma lr alta correria o risco de *"esquecer"* as representações aprendidas no ImageNet.

In [ ]:
# @title Descongelar backbone e configurar otimizador de fine-tuning
for p in base.parameters():
    p.requires_grad = True

optimizer2 = optim.Adam(base.parameters(), lr=1e-4)

trainable2 = sum(p.numel() for p in base.parameters() if p.requires_grad)
print(f'Parâmetros treináveis agora: {trainable2:,}')

In [ ]:
# @title Treinar Fase 2 (5 épocas)
N_EPOCHS_PHASE2 = 5
tr_acc_p2, vl_acc_p2 = run_training(
    base, train_loader, val_loader,
    optimizer2, criterion,
    N_EPOCHS_PHASE2, tag='[Fase 2]'
)
print(f'\nAcurácia de validação ao fim da Fase 2: {vl_acc_p2[-1]:.1%}')

## 6. Resultados

Comparamos a evolução da acurácia de validação nas duas fases e calculamos a acurácia final
no conjunto de **teste** (nunca visto durante o treino).

In [ ]:
# @title Curva de acurácia — Fase 1 vs Fase 2
epochs_p1 = list(range(1, N_EPOCHS_PHASE1 + 1))
epochs_p2 = list(range(N_EPOCHS_PHASE1 + 1, N_EPOCHS_PHASE1 + N_EPOCHS_PHASE2 + 1))

fig, ax = plt.subplots(figsize=(9, 4))

ax.plot(epochs_p1, [a * 100 for a in tr_acc_p1], 'b--o', label='Treino  — Fase 1')
ax.plot(epochs_p1, [a * 100 for a in vl_acc_p1], 'b-o',  label='Val     — Fase 1')
ax.plot(epochs_p2, [a * 100 for a in tr_acc_p2], 'r--s', label='Treino  — Fase 2')
ax.plot(epochs_p2, [a * 100 for a in vl_acc_p2], 'r-s',  label='Val     — Fase 2')

ax.axvline(N_EPOCHS_PHASE1 + 0.5, color='gray', linestyle=':', label='Início do fine-tuning')
ax.set_xlabel('Época')
ax.set_ylabel('Acurácia (%)')
ax.set_title('Transfer Learning no CIFAR-10 — Feature Extraction vs Fine-tuning')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# @title Acurácia final no conjunto de teste
_, test_acc = evaluate(base, test_loader, criterion)
print(f'Acurácia no teste (CIFAR-10): {test_acc:.2%}')

## 7. Predições

Grade 3×3 com imagens do conjunto de teste.  
Título verde = predição correta · vermelho = predição errada.

In [ ]:
# @title Grade 3×3 de predições
base.eval()
sample_imgs, sample_lbls = next(iter(test_loader))

with torch.no_grad():
    preds = base(sample_imgs[:9].to(device)).argmax(dim=1).cpu().numpy()

fig, axes = plt.subplots(3, 3, figsize=(8, 8))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(denormalize(sample_imgs[i]))
    real  = sample_lbls[i].item()
    color = 'green' if preds[i] == real else 'red'
    ax.set_title(
        f'Pred: {CLASS_NAMES[preds[i]]}\nReal: {CLASS_NAMES[real]}',
        color=color, fontsize=9
    )
    ax.axis('off')
plt.suptitle('Predições — verde = correto, vermelho = errado', y=1.01)
plt.tight_layout()
plt.show()